In [53]:
!pip install -q pandas numpy scikit-learn xgboost openai tqdm

In [54]:
!pip install -q groq

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Optional stronger model
from xgboost import XGBClassifier

# LLM Client (choose one provider)
from openai import OpenAI

# -----------------------------------
# API KEY
# -----------------------------------
from groq import Groq

API_KEY = " "

client = Groq(api_key=API_KEY)

MODEL_NAME = "llama-3.1-8b-instant"

In [56]:
# Path to downloaded Lending Club CSV
# Example: "accepted_2007_to_2018Q4.csv"

DATA_PATH = "loan.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Shape:", df.shape)
df.head()

Shape: (396030, 27)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
0,10000.0,36 months,11.44,329.48,B,B4,Marketing,10+ years,RENT,117000.0,...,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0,"0174 Michelle Gateway\r\nMendozaberg, OK 22690"
1,8000.0,36 months,11.99,265.68,B,B5,Credit analyst,4 years,MORTGAGE,65000.0,...,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0,"1076 Carney Fort Apt. 347\r\nLoganmouth, SD 05113"
2,15600.0,36 months,10.49,506.97,B,B3,Statistician,< 1 year,RENT,43057.0,...,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0,"87025 Mark Dale Apt. 269\r\nNew Sabrina, WV 05113"
3,7200.0,36 months,6.49,220.65,A,A2,Client Advocate,6 years,RENT,54000.0,...,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0,"823 Reid Ford\r\nDelacruzside, MA 00813"
4,24375.0,60 months,17.27,609.33,C,C5,Destiny Management Inc.,9 years,MORTGAGE,55000.0,...,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0,"679 Luna Roads\r\nGreggshire, VA 11650"


In [57]:
use_cols = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "pub_rec_bankruptcies",
    "loan_status"
]

df = df[use_cols].copy()

print("Shape:", df.shape)
df.head()

Shape: (396030, 19)


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,dti,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,loan_status
0,10000.0,36 months,11.44,329.48,B,10+ years,RENT,117000.0,Not Verified,vacation,26.24,16.0,0.0,36369.0,41.8,25.0,0.0,0.0,Fully Paid
1,8000.0,36 months,11.99,265.68,B,4 years,MORTGAGE,65000.0,Not Verified,debt_consolidation,22.05,17.0,0.0,20131.0,53.3,27.0,3.0,0.0,Fully Paid
2,15600.0,36 months,10.49,506.97,B,< 1 year,RENT,43057.0,Source Verified,credit_card,12.79,13.0,0.0,11987.0,92.2,26.0,0.0,0.0,Fully Paid
3,7200.0,36 months,6.49,220.65,A,6 years,RENT,54000.0,Not Verified,credit_card,2.60,6.0,0.0,5472.0,21.5,13.0,0.0,0.0,Fully Paid
4,24375.0,60 months,17.27,609.33,C,9 years,MORTGAGE,55000.0,Verified,credit_card,33.95,13.0,0.0,24584.0,69.8,43.0,1.0,0.0,Charged Off


In [58]:
approve_status = ["Fully Paid", "Current"]

deny_status = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)",
    "In Grace Period"
]

df = df[df["loan_status"].isin(approve_status + deny_status)].copy()

df["target"] = df["loan_status"].apply(
    lambda x: 1 if x in approve_status else 0
)

print(df["target"].value_counts())
df.head()

target
1    318357
0     77673
Name: count, dtype: int64


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,dti,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,loan_status,target
0,10000.0,36 months,11.44,329.48,B,10+ years,RENT,117000.0,Not Verified,vacation,26.24,16.0,0.0,36369.0,41.8,25.0,0.0,0.0,Fully Paid,1
1,8000.0,36 months,11.99,265.68,B,4 years,MORTGAGE,65000.0,Not Verified,debt_consolidation,22.05,17.0,0.0,20131.0,53.3,27.0,3.0,0.0,Fully Paid,1
2,15600.0,36 months,10.49,506.97,B,< 1 year,RENT,43057.0,Source Verified,credit_card,12.79,13.0,0.0,11987.0,92.2,26.0,0.0,0.0,Fully Paid,1
3,7200.0,36 months,6.49,220.65,A,6 years,RENT,54000.0,Not Verified,credit_card,2.60,6.0,0.0,5472.0,21.5,13.0,0.0,0.0,Fully Paid,1
4,24375.0,60 months,17.27,609.33,C,9 years,MORTGAGE,55000.0,Verified,credit_card,33.95,13.0,0.0,24584.0,69.8,43.0,1.0,0.0,Charged Off,0


In [59]:
# -------------------------------
# Clean text / percentage columns
# -------------------------------

if "term" in df.columns:
    df["term"] = df["term"].astype(str).str.extract(r"(\d+)").astype(float)

if "int_rate" in df.columns:
    df["int_rate"] = df["int_rate"].astype(str).str.replace("%", "", regex=False).astype(float)

if "revol_util" in df.columns:
    df["revol_util"] = df["revol_util"].astype(str).str.replace("%", "", regex=False)
    df["revol_util"] = pd.to_numeric(df["revol_util"], errors="coerce")

if "emp_length" in df.columns:
    df["emp_length"] = (
        df["emp_length"]
        .astype(str)
        .str.extract(r"(\d+)")
        .fillna(0)
        .astype(float)
    )

# -------------------------------
# Separate X / y
# -------------------------------

X = df.drop(columns=["loan_status", "target"])
y = df["target"]

# -------------------------------
# Detect column types
# -------------------------------

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:", cat_cols)
print("Numeric:", num_cols)

# -------------------------------
# Fill missing values
# -------------------------------

for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

for col in cat_cols:
    X[col] = X[col].fillna("Unknown")

# -------------------------------
# Label encode categoricals
# -------------------------------

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# -------------------------------
# Train test split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Categorical: ['grade', 'home_ownership', 'verification_status', 'purpose']
Numeric: ['loan_amnt', 'term', 'int_rate', 'installment', 'emp_length', 'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies']
Train: (316824, 18)
Test : (79206, 18)


In [60]:
# You can switch between RandomForest and XGBoost

USE_XGBOOST = True

if USE_XGBOOST:
    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42
    )
else:
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )

model.fit(X_train, y_train)

# -------------------------
# Evaluate ML model
# -------------------------
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)

print("ML Model Accuracy:", round(acc, 4))
print()
print(classification_report(y_test, pred))

ML Model Accuracy: 0.8079

              precision    recall  f1-score   support

           0       0.57      0.08      0.14     15535
           1       0.81      0.98      0.89     63671

    accuracy                           0.81     79206
   macro avg       0.69      0.53      0.52     79206
weighted avg       0.77      0.81      0.75     79206



In [ ]:
# ==================================================
# Helper function for Gemini
# ==================================================


import time

def ask_llm(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=0.2,
                max_tokens=300
            )

            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Retry {attempt+1}: {e}")
            time.sleep(1)

    return """
Decision: APPROVE
Confidence: 0.50
Reasoning: Temporary API failure.
"""


# ==================================================
# Convert row to readable applicant profile
# ==================================================

def row_to_profile(row):
    return f"""
Applicant Financial Profile:
- Loan Amount: {row['loan_amnt']}
- Term (months): {row['term']}
- Interest Rate: {row['int_rate']}
- Installment: {row['installment']}
- Grade: {row['grade']}
- Employment Length: {row['emp_length']}
- Home Ownership: {row['home_ownership']}
- Annual Income: {row['annual_inc']}
- Verification Status: {row['verification_status']}
- Purpose: {row['purpose']}
- Debt-to-Income Ratio: {row['dti']}
- Open Accounts: {row['open_acc']}
- Public Records: {row['pub_rec']}
- Revolving Balance: {row['revol_bal']}
- Revolving Utilization: {row['revol_util']}
- Total Accounts: {row['total_acc']}
- Mortgage Accounts: {row['mort_acc']}
- Public Record Bankruptcies: {row['pub_rec_bankruptcies']}
"""


# ==================================================
# Shared Output Format
# ==================================================
FORMAT_RULE = """
Return ONLY in this format:

Decision: APPROVE or DENY
Confidence: number between 0 and 1
Reasoning: your argument
Rebuttal: respond to other agents if context exists, else say None
"""


# ==================================================
# Agent Prompts
# ==================================================

GENERATOR_PROMPT = """
You are Generator Agent.

Role:
Propose an initial lending decision using the applicant profile only.

Focus on:
- overall eligibility
- ability to repay
- general credit strength
- balanced first judgment

Be decisive but rational.
""" + FORMAT_RULE


RISK_PROMPT = """
You are Risk Analyst Agent.

Role:
Evaluate financial risk deeply.

Focus on:
- annual income vs loan amount
- debt burden
- revolving utilization
- public records
- account behavior
- possible default probability

If risk is high, deny.
If risk is manageable, approve.
""" + FORMAT_RULE


POLICY_PROMPT = """
You are Policy Checker Agent.

Role:
Check compliance with responsible lending policies.

Rules:
- Extremely high DTI may violate policy
- Bankruptcy history increases restrictions
- Very low income for large loan may fail policy
- Severe risk signals should be denied

Decide strictly using policy logic.
""" + FORMAT_RULE


EVIDENCE_PROMPT = """
You are Evidence Agent.

Role:
Verify whether approval or denial is supported by evidence in the profile.

Use facts only.
No emotional reasoning.
Mention specific numeric indicators.

Decide based on evidence strength.
""" + FORMAT_RULE


ML_PROMPT_TEMPLATE = """
You are ML Agent.

A trained machine learning model predicted:

Prediction: {pred}
Confidence: {conf}

Your job:
Use this prediction as strong evidence, but explain it in financial terms.

Return in same format.
"""


JUDGE_PROMPT = """
You are Judge Agent.

Role:
Read all agent arguments across all rounds.
Aggregate competing views.
Choose the best final decision.

Requirements:
- consider consensus
- consider strongest evidence
- consider confidence levels
- explain transparently

Return ONLY:

Final Decision: APPROVE or DENY
Final Confidence: number between 0 and 1
Final Reasoning: detailed explanation
"""

In [62]:
import re
import numpy as np

# -----------------------------------------
# Parse agent output
# -----------------------------------------
def parse_agent_output(text):
    decision = "APPROVE"
    confidence = 0.50
    reasoning = text
    rebuttal = "None"

    d = re.search(r"Decision:\s*(APPROVE|DENY)", text, re.I)
    c = re.search(r"Confidence:\s*([0-9]*\.?[0-9]+)", text, re.I)
    r = re.search(r"Reasoning:\s*(.*?)(Rebuttal:|$)", text, re.I | re.S)
    rb = re.search(r"Rebuttal:\s*(.*)", text, re.I | re.S)

    if d:
        decision = d.group(1).upper()

    if c:
        confidence = max(0, min(1, float(c.group(1))))

    if r:
        reasoning = r.group(1).strip()

    if rb:
        rebuttal = rb.group(1).strip()

    return {
        "decision": decision,
        "confidence": confidence,
        "reasoning": reasoning,
        "rebuttal": rebuttal,
        "raw": text
    }


# -----------------------------------------
# Run normal LLM agent
# -----------------------------------------
def run_agent(agent_prompt, row, memory=""):
    profile = row_to_profile(row)

    prompt = f"""
{agent_prompt}

{profile}

Previous Debate Context:
{memory}
"""

    output = ask_llm(prompt)
    return parse_agent_output(output)


# -----------------------------------------
# ML Agent
# -----------------------------------------
def run_ml_agent(row):
    row_df = pd.DataFrame([row]).copy()

    # remove target cols if present
    x = row_df.drop(columns=["loan_status", "target"], errors="ignore")

    # -----------------------------
    # Same cleaning as training
    # -----------------------------
    if "term" in x.columns:
        x["term"] = x["term"].astype(str).str.extract(r"(\d+)").astype(float)

    if "int_rate" in x.columns:
        x["int_rate"] = (
            x["int_rate"].astype(str)
            .str.replace("%", "", regex=False)
            .astype(float)
        )

    if "revol_util" in x.columns:
        x["revol_util"] = (
            x["revol_util"].astype(str)
            .str.replace("%", "", regex=False)
        )
        x["revol_util"] = pd.to_numeric(x["revol_util"], errors="coerce")

    if "emp_length" in x.columns:
        x["emp_length"] = (
            x["emp_length"]
            .astype(str)
            .str.extract(r"(\d+)")
            .fillna(0)
            .astype(float)
        )

    # -----------------------------
    # Fill missing values
    # -----------------------------
    for col in x.columns:
        if col in X_train.columns:
            if col in encoders:
                x[col] = x[col].fillna("Unknown").astype(str)
            else:
                x[col] = pd.to_numeric(x[col], errors="coerce")
                x[col] = x[col].fillna(X_train[col].median())

    # -----------------------------
    # Encode categoricals
    # -----------------------------
    for col, le in encoders.items():
        if col in x.columns:
            val = str(x[col].iloc[0])

            # unseen category handling
            if val not in le.classes_:
                val = le.classes_[0]

            x[col] = le.transform([val])

    # reorder columns
    x = x[X_train.columns]

    # force numeric
    x = x.astype(float)

    # -----------------------------
    # Predict
    # -----------------------------
    proba = model.predict_proba(x)[0]
    pred = int(np.argmax(proba))

    decision = "APPROVE" if pred == 1 else "DENY"
    conf = float(np.max(proba))

    prompt = ML_PROMPT_TEMPLATE.format(
        pred=decision,
        conf=round(conf, 3)
    )

    output = ask_llm(prompt + "\n\n" + row_to_profile(row))
    parsed = parse_agent_output(output)

    parsed["decision"] = decision
    parsed["confidence"] = conf

    return parsed

In [63]:
def compute_uncertainty(agent_results):
    decisions = [a["decision"] for a in agent_results]

    approve_count = decisions.count("APPROVE")
    deny_count = decisions.count("DENY")

    majority = "APPROVE" if approve_count >= deny_count else "DENY"

    agree_conf = [
        a["confidence"]
        for a in agent_results
        if a["decision"] == majority
    ]

    avg_conf = float(np.mean(agree_conf)) if agree_conf else 0.5
    uncertainty = 1 - avg_conf

    return majority, avg_conf, uncertainty


def debate_system(row, verbose=True):
    memory = ""
    rounds = []

    for r in range(1, 3):   # change to range(1,3) for fewer API calls
        round_outputs = {}

        round_outputs["Generator"] = run_agent(GENERATOR_PROMPT, row, memory)
        round_outputs["Risk"] = run_agent(RISK_PROMPT, row, memory)
        round_outputs["Policy"] = run_agent(POLICY_PROMPT, row, memory)
        round_outputs["Evidence"] = run_agent(EVIDENCE_PROMPT, row, memory)
        round_outputs["ML"] = run_ml_agent(row)

        rounds.append(round_outputs)

        # Build memory for next round
        memory = f"Round {r} Summary:\n"

        for name, out in round_outputs.items():
            memory += f"""
{name} Agent
Decision: {out['decision']}
Confidence: {out['confidence']:.2f}
Reasoning: {out['reasoning']}
Rebuttal: {out.get('rebuttal', 'None')}
"""

        # Show debate
        if verbose:
            print("=" * 60)
            print(f"ROUND {r}")
            print("=" * 60)

            for name, out in round_outputs.items():
                print(f"{name} Agent")
                print("Decision   :", out["decision"])
                print("Confidence :", round(out["confidence"], 2))
                print("Reasoning  :", out["reasoning"])
                print("Rebuttal   :", out.get("rebuttal", "None"))
                print("-" * 60)

    # Final uncertainty
    final_agents = list(rounds[-1].values())
    majority, avg_conf, uncertainty = compute_uncertainty(final_agents)

    # Build judge context
    judge_context = ""

    for i, rd in enumerate(rounds, 1):
        judge_context += f"\n===== ROUND {i} =====\n"

        for name, out in rd.items():
            judge_context += f"""
{name} Agent
Decision: {out['decision']}
Confidence: {out['confidence']:.2f}
Reasoning: {out['reasoning']}
Rebuttal: {out.get('rebuttal', 'None')}
"""

    judge_output = ask_llm(JUDGE_PROMPT + "\n\n" + judge_context)

    # Parse judge
    final_decision = majority
    final_conf = avg_conf
    final_reasoning = judge_output

    d = re.search(r"Final Decision:\s*(APPROVE|DENY)", judge_output, re.I)
    c = re.search(r"Final Confidence:\s*([0-9]*\.?[0-9]+)", judge_output, re.I)
    rr = re.search(r"Final Reasoning:\s*(.*)", judge_output, re.I | re.S)

    if d:
        final_decision = d.group(1).upper()

    if c:
        final_conf = float(c.group(1))

    if rr:
        final_reasoning = rr.group(1).strip()

    result = {
        "decision": final_decision,
        "confidence": final_conf,
        "uncertainty": uncertainty,
        "majority": majority,
        "rounds": rounds,
        "reasoning": final_reasoning
    }

    if verbose:
        print("=" * 60)
        print("FINAL RESULT")
        print("=" * 60)
        print("Final Decision :", result["decision"])
        print("Confidence     :", round(result["confidence"], 3))
        print("Uncertainty    :", round(result["uncertainty"], 3))
        print("Majority Vote  :", result["majority"])
        print()
        print("Judge Explanation:")
        print(result["reasoning"])

    return result

In [ ]:
# Pick one sample from test set
sample_idx = X_test.index[0]

sample_row = df.loc[sample_idx]

result = debate_system(sample_row, verbose=True)

ROUND 1
Generator Agent
Decision   : DENY
Confidence : 0.8
Reasoning  : The applicant has a low credit grade (D), a high debt-to-income ratio (22.62%), and a high revolving balance ($32,484.00) with a utilization rate of 52.7%. These factors indicate a high risk of default. Additionally, the loan amount is substantial ($25,000.00) and the interest rate is relatively high (14.61%). The applicant's employment length is 9 months, which is relatively short, and their home ownership status is rent, which may indicate a lack of long-term financial stability. The verification status is also not verified, which adds to the risk.
Rebuttal   : None
------------------------------------------------------------
Risk Agent
Decision   : DENY
Confidence : 0.8
Reasoning  : The applicant's debt-to-income ratio is 22.62%, which is relatively high. Additionally, they have a high revolving utilization of 52.7% and a large number of open accounts (15). Their credit grade is also D, indicating poor credit hi